# Chapter 3: Error Analysis
Steps: make test queries → run the system → read traces (open coding) → group notes (axial coding) → label and count (prevalence).

In [ ]:
from common import ask, load_data
import itertools, random
import pandas as pd

data = load_data()["ch3"]

## 1. Synthetic queries: dimensions → tuples → queries
**Dimensions** = ways requests can differ. A **tuple** = one value from each dimension.

In [ ]:
dims = data["dimensions"]
for name, values in dims.items():
    print(name, "->", values)

all_tuples = list(itertools.product(*dims.values()))
print("\nTotal possible tuples:", len(all_tuples))

random.seed(1)
chosen = random.sample(all_tuples, 5)           # pick 5 to test
for t in chosen:
    print(t)

Now ask the LLM to write one realistic user message per tuple.

In [ ]:
queries = []
for feature, persona, scenario in chosen:
    prompt = (data["query_prompt"]
              .replace("{feature}", feature)
              .replace("{client_persona}", persona)
              .replace("{scenario}", scenario))
    q = ask(prompt, temperature=0.7)
    queries.append(q)
    print(f"({feature}, {persona}, {scenario})\n  -> {q}\n")

## 2. Look at traces
A **trace** = the full record of one run (query + response; real apps also have tool calls). We use 10 ready-made traces from the JSON file.

In [ ]:
traces = data["traces"]
for t in traces:
    print(f"#{t['id']} Q: {t['query']}\n    A: {t['response']}\n")

## 3. Open coding
Read each trace and write a short note about the **first thing that went wrong**. Empty note = no problem.

In [ ]:
for t in traces:
    print(f"#{t['id']:<3} note: {t['open_note'] or '(no failure)'}")

## 4. Axial coding
Group similar notes under one **failure mode** name (this list is your *taxonomy*).

In [ ]:
axial_map = data["axial_map"]       # note -> failure mode
for note, mode in axial_map.items():
    print(f"{note:40} -> {mode}")

taxonomy = sorted(set(axial_map.values()))
print("\nTaxonomy:", taxonomy)

## 5. Labeling table + prevalence
One column per failure mode: `1` = happened, `0` = did not. **Prevalence** = traces with failure ÷ all traces.

In [ ]:
rows = []
for t in traces:
    mode = axial_map.get(t["open_note"])             # None if no failure
    row = {"trace": t["id"]}
    for m in taxonomy:
        row[m] = 1 if mode == m else 0
    rows.append(row)

labels = pd.DataFrame(rows).set_index("trace")
display(labels)

prevalence = labels.mean().sort_values(ascending=False)
print("Prevalence (share of traces):")
print((prevalence * 100).round(0).astype(int).astype(str) + "%")

## 6. Saturation check
Walk through the traces one by one. When do **new** failure modes stop appearing?

In [ ]:
seen = set()
for t in traces:
    mode = axial_map.get(t["open_note"])
    if mode and mode not in seen:
        seen.add(mode)
        print(f"trace #{t['id']}: NEW failure mode -> {mode}")
print("\nTotal modes found:", len(seen), "| If the last 20 traces show nothing new, you are done (book suggests ~100 traces).")